<table style="background-color:#F5F5F5;" width="100%">
<tr><td style="background-color:#F5F5F5;"><img src="../images/logo.png" width="150" align='right'/></td></tr>     <tr><td>
            <h2><center>Aprendizagem Automática em Engenharia Biomédica</center></h2>
            <h3><center>1st Semester - 2026/2027</center></h3>
            <h4><center>Universidade Nova de Lisboa - Faculdade de Ciências e Tecnologia</center></h4>
</td></tr>
    <tr><td><h2><b><center>Lab 7 - Model Optimization: Hyperparameter Tuning</center></b></h2>
    <h4><i><b><center>Human Activity Recognition with Decision Trees</center></b></i></h4></td></tr>
</table>

## 1. Introduction to Optimizing Machine Learning Models

In machine learning, building a predictive model is only the first step toward solving a real-world problem. To ensure the model performs well in practice, it is essential to optimize it by finding the best combination of model parameters and structure. This process of __model optimization__ helps us enhance a model's ability to generalize, meaning it performs well on new, unseen data.

### 1.1 The Importance of Model Optimization

Model optimization allows us to achieve three key goals:

* __Generalization__: A well-optimized model should generalize effectively across various datasets. Generalization refers to the model’s ability to perform well not just on the training data but also on new data it hasn't seen before.
* __Model Selection__: Optimization allows us to systematically compare different models and select the one that performs best for the task at hand.
* __Avoiding Overfitting and Underfitting__:
    * __Overfitting__: occurs when a model learns the noise in the training data instead of the underlying patterns. This results in high performance on training data but poor performance on new data.
    * __Underfitting__: happens when the model is too simple to capture the patterns in the data, resulting in low performance on both training and testing data.

### 1.2 Goal of Today's Class

In today's practical lession we are going to use two specific strategies that can be employed for model optimization. These are

* __Cross Validation__: a technique for assessing a model's performance on different substes of the data.
* __Hyperparemeter Tuning__: a technique to adapt the model's external parameters to improve the data fit.

Together, cross validation and hyperparameter tuning form a powerful toolkit for model optimization. Cross validation ensures that the model is robust and generalizes well, while hyperparameter tuning tailors the model’s complexity and behavior to the data, enhancing its overall performance.

### 1.3 Revisiting the UCI Human Activity Recognition Dataset

As always, to train models we need data. To simplify things, we will use the same dataset that we used in the previous lesson: [__the UCI Human Activity Recognition Using Smartphones Dataset__](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones).

The code below loads the data, windows it, and extracts features the same way we did in the previous class, __with the exception that the window_splitter() function also retrieves the subject_id for the corresponding window__.

In [1]:
# -- imports needed for the Notebook -- #

%matplotlib notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import Tuple, List, Union, Any, Dict

In [2]:
# -- definition of functions -- #

def get_inertial_data(acc_axis: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Loads and processes the inertial signal data from the UCI Human Activity Recognition (HAR) dataset.

    This function reads accelerometer data for a specified axis (e.g., 'x', 'y', or 'z') for both 
    the training and test datasets. It also loads corresponding activity labels and subject IDs. 
    The data is windowed and repeated 64 times to match the window size.

    :param acc_axis: The axis of accelerometer data to load ('x', 'y', or 'z').

    :returns: 
        - X_train: Training set accelerometer data (horizontally stacked).
        - y_train: Repeated activity labels for the training set.
        - subject_train: Repeated subject IDs for the training set.
        - X_test: Test set accelerometer data (windowed and stacked horizontally).
        - y_test: Repeated activity labels for the test set.
        - subject_test: Repeated subject IDs for the test set.
        - activity_labels: Array of activity labels for both training and test sets.

    :rtype: Tuple of NumPy arrays containing processed data.
    """
    # --- loading training data --- #
    # Load training accelerometer data for the specified axis
    X_train = np.loadtxt('Data/UCI HAR Dataset/train/Inertial Signals/total_{}_train.txt'.format(acc_axis))

    # stack all data horizontally to obtain a 1D array
    X_train = np.hstack([window[:64] for window in X_train])
    
    # Load activity labels for training data
    y_train = np.loadtxt('Data/UCI HAR Dataset/train/y_train.txt', dtype=int)

    # repeat 64 times to have a label for each training data sample
    y_train = np.repeat(y_train, 64)
    
    # Load subject IDs for training data
    subject_train = np.loadtxt('Data/UCI HAR Dataset/train/subject_train.txt', dtype=int)

    # repeat 64 times to have a subject ID for each training data sample
    subject_train = np.repeat(subject_train, 64)

    # --- loading testing data --- #
    # Load test accelerometer data for the specified axis
    X_test = np.loadtxt('Data/UCI HAR Dataset/test/Inertial Signals/total_{}_test.txt'.format(acc_axis))

    # stack all data horizontally to obtain a 1D array
    X_test = np.hstack([window[:64] for window in X_test])
    
    # Load activity labels for test data
    y_test = np.loadtxt('Data/UCI HAR Dataset/test/y_test.txt', dtype=int)

    # repeat 64 times to have a label for each testing data sample
    y_test = np.repeat(y_test, 64)
    
    # Load subject IDs for test data
    subject_test = np.loadtxt('Data/UCI HAR Dataset/test/subject_test.txt', dtype=int)

    # repeat 64 times to have a subject ID for each training data sample
    subject_test = np.repeat(subject_test, 64)
    
    # Load activity labels (activity number and corresponding activity name)
    activity_labels = pd.read_csv('Data/UCI HAR Dataset/activity_labels.txt', 
                                  index_col=0, header=None, sep=' ')
    
    # Return all the processed data
    return (X_train, y_train, subject_train, X_test, y_test, subject_test, np.hstack(activity_labels.values))


def window_splitter(acc: np.ndarray, labels: np.ndarray, 
                    subject_ids: np.ndarray, window_size: int, fs: int) -> Tuple[List[np.ndarray], List[int], List[int]]:
    """
    Splits time-series data into non-overlapping windows and assigns labels to each window. Only windows containing a unique label are returned.
    
    :param acc: np.ndarray
        Time-series acceleration data of shape (n_samples,).
    :param labels: np.ndarray
        Array of labels corresponding to each time step in `acc`, with shape (n_samples,).
    :param labels: np.ndarray
        Array of containing the subject ID corresponding to each time step in `acc`, with shape (n_samples,).
    :param window_size: int
        Desired window length in seconds.
    :param fs: int
        Sampling frequency in Hz, used to convert `window_size` to samples.
        
    :returns: Tuple[List[np.ndarray], List[int]]
        A tuple containing:
        - List containing the extracted windows. The list is of shape (n_windows, samples_per_window).
        - List containing the labels assigned to each window.
        - numpy.ndarray containing the subject ID for each window.
    """
    
    # Input validation
    if window_size <= 0 or fs <= 0:
        raise ValueError("Both `window_size` and `fs` must be positive integers.")
    if acc.shape[0] != labels.shape[0]:
        raise ValueError("The `acc` and `labels` arrays must have the same length along the first dimension.")
    
    # Convert window size from seconds to samples
    window_size_samples = window_size * fs
    
    # Initialize lists to hold windows and corresponding labels
    X_windows = []
    y_windows = []
    subject_id_windows = []

    # Sliding window approach
    for w_start in range(0, len(acc) - window_size_samples + 1, window_size_samples):
        current_window_labels = labels[w_start:w_start + window_size_samples]
        
        # Check if the window contains only a single label
        unique_labels = np.unique(current_window_labels)

        # check if the window only contains one unique label (i.e., the entire window contains only data from one activity)
        if len(unique_labels) == 1:

            # add the windows to the list
            y_windows.append(unique_labels[0])
            X_windows.append(acc[w_start:w_start + window_size_samples])
            subject_id_windows.append(subject_ids[w_start])

    return X_windows, y_windows, np.asarray(subject_id_windows)


def compute_statistical_features(X_windows: Union[np.ndarray, list]) -> np.ndarray:
    """
    Computes statistical features (mean, maximum, minimum) for each window in a time-series dataset.
    
    :param X_windows: Union[np.ndarray, list]
        A 2D array or list containing the extracted windows containing the accelerometer data. The list is of shape (number_of_windows, samples_per_window).
        
    :returns: np.ndarray
        A 2D array of shape (n_windows, 3), where each row contains the mean, max, and min of each window.
        
    :raises ValueError: If `X_windows` is not a 2D array or list, or if it has less than 2 columns.
    """
    
    # Convert input to numpy array if it's a list
    X_windows = np.asarray(X_windows)
    
    # Input validation
    if X_windows.ndim != 2:
        raise ValueError("Input `X_windows` must be a 2D array or list of shape (n_windows, n_features).")
    if X_windows.shape[1] < 1:
        raise ValueError("Each window in `X_windows` must contain at least one feature.")
    
    # Compute statistical features for each window
    mean_feat = np.mean(X_windows, axis=1)  # Mean of each window
    max_feat = np.max(X_windows, axis=1)    # Maximum of each window
    min_feat = np.min(X_windows, axis=1)    # Minimum of each window
    
    # Stack features column-wise to form the output array
    statistical_features = np.column_stack((mean_feat, max_feat, min_feat))
    
    return statistical_features

In [3]:
# -- loading, windowing, and feature extraction -- #

# define sampling frequency and window size as constants
FS = 50
W_SECONDS = 5

# (1) load the data
train_data, train_labels, subjects_train, test_data, test_labels, subjects_test, activity_labels = get_inertial_data('acc_x')

# print out some information on the loaded data
print("ACC train data shape: {} | target shape: {} | Subjects: {}".format(train_data.shape, train_labels.shape, np.unique(subjects_train).size))
print("ACC test  data shape: {} | target shape: {} | Subjects: {}".format(test_data.shape, test_labels.shape, np.unique(subjects_test).size))
print("\nAvailable Activities:\n{}".format(activity_labels))

# (2) window the training and the test data
train_windows, y_train, subject_ids_train = window_splitter(train_data, train_labels, subjects_train, W_SECONDS, FS)
test_windows, y_test, subject_ids_test = window_splitter(test_data, test_labels, subjects_test, W_SECONDS, FS)

# (3) compute statistical features on the training and test data
X_train = compute_statistical_features(train_windows)
X_test = compute_statistical_features(test_windows)

# printing the first 10 rows of the training data
print("\nThe first 10 rows of the training data: \n{}".format(X_train[:10]))

ACC train data shape: (470528,) | target shape: (470528,) | Subjects: 21
ACC test  data shape: (188608,) | target shape: (188608,) | Subjects: 9

Available Activities:
['WALKING' 'WALKING_UPSTAIRS' 'WALKING_DOWNSTAIRS' 'SITTING' 'STANDING'
 'LAYING']

The first 10 rows of the training data: 
[[1.01958714 1.027664   1.009013  ]
 [1.02032754 1.026194   1.013645  ]
 [1.02092431 1.02821    1.01161   ]
 [1.01553708 1.025468   0.9974581 ]
 [1.00990811 1.015505   1.002174  ]
 [1.0097102  1.015896   1.003571  ]
 [0.92792139 0.9403026  0.90814   ]
 [0.80211137 1.069117   0.6960658 ]
 [0.82745595 0.968194   0.7706567 ]
 [0.95521952 0.9630642  0.9459184 ]]


### 2. Hyperparameter Optimization

Hyperparameter optimization is the process of systematically searching for the best hyperparameters to improve a model’s performance. Hyperparameters are parameters set before the learning process begins, controlling aspects like model complexity and training behavior. Unlike model parameters (e.g., weights in a neural network), hyperparameters do not directly interact with the data during training, so they need to be tuned manually or through automated methods.

Selecting the optimal hyperparameters can greatly impact a model's effectiveness, balancing the trade-off between bias and variance to prevent underfitting and overfitting.

### 2.1. Purpose of Hyperparameter Optimization

Hyperparameter optimization is essential because:

* __Improves Model Performance__: Finding the optimal configuration helps maximize the model’s predictive accuracy.
* __Prevents Overfitting/Underfitting__: Tuning hyperparameters helps the model generalize better on unseen data, finding the right balance between bias and variance.
* __Efficient Resource Usage__: Optimization techniques allow us to find good settings without an exhaustive search, saving time and computational resources.

### 2.2 Hyperparameter Optimization Methods

There are several popular methods for hyperparameter optimization, each with unique strategies to explore the parameter space. Here, we cover three widely-used approaches: Grid Search, Random Search, and Bayesian Search.

__1. Grid Search__:
* __Overview__: Grid Search is an exhaustive search method that tries every possible combination of specified hyperparameter values. For each combination, cross-validation is performed to evaluate model performance, and the best configuration is selected.
* __Advantages__: Provides a thorough exploration of the hyperparameter space, ensuring that all combinations are tested.
* __Disadvantages__: Computationally expensive, especially for large parameter spaces. Often impractical when the number of hyperparameters or their ranges is large.
* __Use Case__: Useful when you have a relatively small number of hyperparameters or can limit each parameter to a small set of values.
* __sklearn function__: [__GridSearchCV()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

__2. Random Search__:
* __Overview__: Instead of trying all combinations, Random Search randomly samples a specified number of hyperparameter configurations from the search space. This approach is faster than Grid Search as it explores fewer combinations, yet it can still yield high-performing models.
* __Advantages__: More efficient than Grid Search, as it focuses on sampling a diverse range of configurations rather than every possible combination.
* __Disadvantages__: There’s no guarantee of finding the absolute best combination, as some configurations may be missed.
* __Use Case__: Ideal for optimizing models with many hyperparameters or when each model evaluation is time-consuming, such as deep learning tasks or complex ML pipelines.
* __sklearn function__: [__RandomizedSearchCV()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV)

__3. Bayesian Optimization__:
* __Overview__: Bayesian Optimization is a probabilistic model-based approach that builds a surrogate model to predict the performance of various hyperparameter combinations. It iteratively refines this model to focus the search on promising areas of the parameter space.
* __Advantages__: Efficient for high-dimensional search spaces, as it focuses the search on likely optimal regions instead of covering the entire space. Reduces the number of evaluations required.
* __Disadvantages__: Requires more setup and computational complexity than Grid or Random Search but generally achieves better results in fewer iterations.
* __Use Case__: Effective when the search space is large or when certain hyperparameters have little impact on performance, allowing Random Search to efficiently cover a wide variety of configurations.
* __sklearn function__: [__BayesSearchCV()__](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html)

### 2.3 Choosing an Optimization Method
The choice of optimization method depends on factors like the complexity of the model, the number of hyperparameters, and the available computational resources. For simpler models or smaller search spaces, Grid Search or Random Search may be sufficient. However, for larger or more complex search spaces, Bayesian Optimization provides a more efficient, data-driven approach.

Hyperparameter optimization is a powerful tool in model tuning, allowing us to maximize performance by systematically searching for the best configuration of settings. In the following exercises, we will explore how to apply these techniques to improve the accuracy and generalization of our models.


### 2.4 Implementing Random Search using Scikit-learn

We will optimize a Decision Tree using `RandomizedSearchCV`. The search evaluates candidate hyperparameter combinations with cross-validation. We use `GroupKFold` because multiple windows can come from the same participant: all windows from one participant must stay in the same fold when we want to estimate performance on unseen participants.

**Important:** keep the test set separate. We use only the training set for tuning; the test set is used for the final evaluation after the choices are made.

__Exercise 1__:

Before running the search, inspect the `param_grid` below.

**Tasks**
1. How many possible hyperparameter combinations are defined?
2. What do you expect to happen to training accuracy when `max_depth` increases? What about validation accuracy?

**Hint:** the total number of combinations is the product of the number of candidate values for each hyperparameter.

In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

# Reproducibility and search budget
rand_state = 42
iterations = 100  # Increase if you have more time/computing resources

# Define the model and the cross-validation strategy
dt_clf = DecisionTreeClassifier(random_state=rand_state)
group_kfold = GroupKFold(n_splits=10)

# Candidate values for each hyperparameter
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': range(2, 6),
    'min_samples_split': [50, 100, 200],
    'min_samples_leaf': [25, 50, 100],
    'max_leaf_nodes': [10, 20, 30]
}

# RandomizedSearchCV samples combinations from the parameter distributions
optimized_dt_clf = RandomizedSearchCV(
    estimator=dt_clf,
    param_distributions=param_grid,
    cv=group_kfold,
    scoring='accuracy',
    n_iter=iterations,
    random_state=rand_state,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)

In [5]:
# Add solution here

After having defined the optimization method, it is necessary to use the `fit()` method to optimize the hyperparameters. Doing this will train and evaluate all the models (with their different hyperparameters) on the respective folds. 

In [6]:
# Pass subject IDs so GroupKFold keeps each participant in one fold
optimized_dt_clf.fit(X_train, y_train, groups=subject_ids_train)

,estimator,DecisionTreeC...ndom_state=42)
,param_distributions,"{'criterion': ['gini', 'entropy'], 'max_depth': range(2, 6), 'max_leaf_nodes': [10, 20, ...], 'min_samples_leaf': [25, 50, ...], ...}"
,n_iter,100
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,GroupKFold(n_...shuffle=False)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
# get the score of the best model
# the best_score is the mean accuracy over all folds
print("The best score/accuracy is: {}".format(optimized_dt_clf.best_score_))

The best score/accuracy is: 0.7339557270960961


In [8]:
# display the parameters of the best estimator
print("The best model, together with its hyperparameter settings, is: \n{}".format(optimized_dt_clf.best_estimator_))

The best model, together with its hyperparameter settings, is: 
DecisionTreeClassifier(criterion='entropy', max_depth=3, max_leaf_nodes=30,
                       min_samples_leaf=100, min_samples_split=50,
                       random_state=42)


__Exercise 2__:

`RandomizedSearchCV` stores information about every candidate in `cv_results_`.

**Tasks**
1. Create a DataFrame from `optimized_dt_clf.cv_results_` and display the columns `params`, `mean_test_score`, `std_test_score`, and `mean_train_score`.
2. Show the 10 best candidates ranked by `mean_test_score`.
3. Compare the best and worst mean validation scores.
4. Do the top candidates have very different scores, or are several configurations similarly good? What does this suggest about choosing a single “best” configuration?

__Exercise 3__:

**Tasks**
1. Add a column called `train_validation_gap` equal to `mean_train_score - mean_test_score`.
2. Display the five candidates with the largest gap.
3. Compare their training and validation scores. Do high training scores always correspond to high validation scores?
4. Explain why selecting the model with the highest training accuracy would be a poor tuning strategy.


__Exercise 4:__ 

From this, we now know the best combination of hyperparameters (among the 100 runs) that optimizes the model towards the 10 validation sets from cross validation. Nevertheless, it is still important to validate the generalization capabilities of our model on the left out test set.

__Tasks__:
* Evaluate the results of the optimized classifier using the test and the training data.
* Use scikit-learn's [__classification report()__](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) function to generate a comprehensive summary of key metrics for evaluating a classification model's performance with regards to each class.
*  Discuss the results, namely by comparing between different classes and the aggregated ones. How they compare to the results from the train set?

__Hints__:
* Pass the activity_labels to the `target_names` parameter.

__Exercise 5:__ 
Next, compare the statistical features with a larger feature set extracted using TSFEL. 

__Tasks__:
* Extract features for the train and test data using TSFEL.
* Optimize a Decision Tree model resorting to [__GridSearchCV()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) and using a [__LeaveOneGroupOut()__](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html)
* As these data sets now includes more features, you should limit the number of features leveraged by the model, as they come at a cost of complexity. The amount of features should be defined in the hyperparamter grid. 

__Hints__: 
* This exercise can take considerably longer because TSFEL creates many features and Leave-One-Group-Out creates one split per subject. Start with a small grid, then expand if time permits. Try using the `n_jobs` option, which will distribute the computations across multiple cores.
* Take a look at the `max_features` hyperparameter from the Decision Tree estimator, to limit the amount of features you are passing

In [ ]:
import tsfel

# get all features from TSFEL
cfg = tsfel.get_features_by_domain()

# extract features for the training and test data
X_train_tsfel = tsfel.time_series_features_extractor(cfg, train_windows, fs=50)
X_test_tsfel = tsfel.time_series_features_extractor(cfg, test_windows, fs=50)

__Exercise 6:__ Evaluate the TSFEL model on the held-out test set and compare it with the model trained on statistical features. Discuss the scores, per-class performance, feature-set size, and computational cost. Avoid claiming that one feature set is universally better based on this one split.